In [ ]:
# 1. Подготовка корпуса данных.

import datasets
from datasets import load_dataset
import tokenizers
import transformers
from transformers import AutoTokenizer
import re
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Sequence, Whitespace, Punctuation
from tokenizers.trainers import BpeTrainer
from pathlib import Path
import pandas as pd
from IPython.display import display

# 1. Подготовка корпуса данных.

#  подгрузили датасет и посмотрели данные
dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-ru",
    split="train"
)

print(dataset)
print(dataset[0])

Dataset({
    features: ['id', 'translation'],
    num_rows: 17496
})
{'id': '0', 'translation': {'en': 'Anna Karenina', 'ru': 'Анна Каренина'}}


In [3]:
# базовая очистка текста от пробелов и пустых строк
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    return text


# выполняем требования создать "Python-генератор (iterator)" экономии памяти (return вместо yield был бы менее экономный)
def text_generator(dataset):
    for item in dataset:
        text = item["translation"]["ru"]
        text = clean_text(text)

        if text:
            yield text

generator = text_generator(dataset)

# через enumerate перебираем 5 первых строк получая одновременно и строки, и индексы
for i, text in enumerate(generator):
    print(text)
    
    if i == 4:
        break

Анна Каренина
Толстой Лев Николаевич
Мне отмщение, и аз воздам
ЧАСТЬ ПЕРВАЯ
I


In [4]:
# 2. Проектирование и обучение токенизатора.

# создали пустой BPE
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# настройка токенизации - Pre-tokenizers
tokenizer.pre_tokenizer = Sequence([
    Whitespace(),
    Punctuation(),
])

# настраиваем trainer
trainer = BpeTrainer(
    vocab_size=20000,  # размер словаря в токенах
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] # резервация специальных токенов
)

In [5]:
# Обучаем BPE на генераторе, создаем генаратор заново т.к. уже часть данных выше перебирали тестово

generator = text_generator(dataset)
tokenizer.train_from_iterator(
    generator,
    trainer=trainer
)

# проверяем размер словаря после обучения, смотрим самые длинные токены, т.е. они не разбились на куски
print("Размер словаря:", tokenizer.get_vocab_size())

# демонстраия, что выучил BPE. 
vocab = tokenizer.get_vocab()
long_tokens = sorted(
    vocab.items(),
    key=lambda x: len(x[0]),
    reverse=True
)

for token, token_id in long_tokens[:5]:
    print(token_id, repr(token))

Размер словаря: 20000
16616 'достопримечательно'
7024 'превосходительство'
16434 'последовательность'
9476 'усовершенствования'
15011 'действительностью'


In [6]:
# еще немного демонстрации, как токенезировались длинные слова, тут уже видим, 
# что длинное слово разбилось более мелкие составляющие
encoded = tokenizer.encode("сельскохозяйственный")

print("Токены:", encoded.tokens)
print("ID:", encoded.ids)

Токены: ['сель', 'ско', 'хозяйстве', 'нный']
ID: [3719, 234, 2695, 647]


In [7]:
# завершаем п.2 ДЗ сохранением словаря в json
tokenizer.save("custom_russian_bpe.json")

# проверяем, что файл сохранен
tokenizer_path = Path("custom_russian_bpe.json")
print("Файл существует:", tokenizer_path.exists())
print("Размер:", tokenizer_path.stat().st_size, "байт")

Файл существует: True
Размер: 1615750 байт


In [8]:
# 3. Анализ слабых мест и тестирование морфологии.

# берем согласно ДЗ уже обученный токенезатор "cointegrated/rubert-tiny2" для сравнения с тем, что обучили сами
industrial_tokenizer = AutoTokenizer.from_pretrained(
    "cointegrated/rubert-tiny2"
)
print("Размер словаря:", industrial_tokenizer.vocab_size)

Размер словаря: 83828


In [9]:
# тестовая выборка по которой будем сравнивать токенизаторы
test_texts = [
    "Мы переосмысливаем недопонятые результаты эксперимента.",
    "Сельскохозяйственный и железнодорожный комплексы требуют модернизации.",
    "Я вчера загуглил это, но ответ оказался каким-то кринжовым.",
    "Достоевский и Шолохов остаются важными фигурами русской литературы.",
    "Нейросетевой автопереводчик неожиданно переинтерпретировал исходное сообщение."
]

# прогоняем тестовую выборку через само обученный токенизатор и через готовое индустриальное решение
for text in test_texts:
    my_tokens = tokenizer.encode(text).tokens
    industrial_tokens = industrial_tokenizer.tokenize(text)

    print("=" * 80)
    print("Текст:", text)

    print("\nНаш BPE:")
    print(my_tokens)

    print("\nRuBERT:")
    print(industrial_tokens)

# Сравнение: 
# - имена собственные (Достоевский, Шолохов) обработались индустриальным токенезатором заметно лучше
# - простые предложение разбиты на токены само обученным токенезатором сравнительно не плохо, но у индустриального больше длинных слов (меньше токенов)
# - современные и сленговые слова плохо обработаны во всех случаях (пример "заг"углил" и "кринжовым")

Текст: Мы переосмысливаем недопонятые результаты эксперимента.

Наш BPE:
['Мы', 'пере', 'ос', 'мысли', 'ваем', 'недо', 'поня', 'тые', 'результа', 'ты', 'эк', 'спе', 'ри', 'мента', '.']

RuBERT:
['Мы', 'переос', '##мысли', '##ваем', 'недо', '##пон', '##ят', '##ые', 'результаты', 'эксперимента', '.']
Текст: Сельскохозяйственный и железнодорожный комплексы требуют модернизации.

Наш BPE:
['С', 'ель', 'ско', 'хозяйстве', 'нный', 'и', 'железнодоро', 'жный', 'ком', 'п', 'лек', 'сы', 'требу', 'ют', 'мо', 'дер', 'ни', 'за', 'ции', '.']

RuBERT:
['Сель', '##скохозяй', '##ственный', 'и', 'железнодорожный', 'комплексы', 'требуют', 'модернизации', '.']
Текст: Я вчера загуглил это, но ответ оказался каким-то кринжовым.

Наш BPE:
['Я', 'вчера', 'загу', 'гли', 'л', 'это', ',', 'но', 'ответ', 'оказался', 'каким', '-', 'то', 'кри', 'н', 'жо', 'вым', '.']

RuBERT:
['Я', 'вчера', 'загу', '##гли', '##л', 'это', ',', 'но', 'ответ', 'оказался', 'каким', '-', 'то', 'кри', '##н', '##жо', '##вым', '.']
Текст: 

In [ ]:
# 4. Расчёт базовых метрик (Оценка эффективности).

# Считаем число слов без пунктуации
def count_words(text):
    return len(re.findall(r"[а-яёА-ЯЁa-zA-Z0-9]+", text))


my_ratios = []
rubert_ratios = []

# Считаем токен/слово и сохраняем результа в списки
for text in test_texts:
    words = count_words(text)

    my_tokens = tokenizer.encode(text).tokens
    rubert_tokens = industrial_tokenizer.tokenize(text)

    my_ratio = len(my_tokens) / words
    rubert_ratio = len(rubert_tokens) / words

    my_ratios.append(my_ratio)
    rubert_ratios.append(rubert_ratio)

    print("=" * 70)
    print(text)
    print("Слов:", words)
    print("Наш BPE:", len(my_tokens), "токенов →", round(my_ratio, 2), "токена/слово")
    print("RuBERT:", len(rubert_tokens), "токенов →", round(rubert_ratio, 2), "токена/слово")

Мы переосмысливаем недопонятые результаты эксперимента.
Слов: 5
Наш BPE: 15 токенов → 3.0 токена/слово
RuBERT: 11 токенов → 2.2 токена/слово
Сельскохозяйственный и железнодорожный комплексы требуют модернизации.
Слов: 6
Наш BPE: 20 токенов → 3.33 токена/слово
RuBERT: 9 токенов → 1.5 токена/слово
Я вчера загуглил это, но ответ оказался каким-то кринжовым.
Слов: 10
Наш BPE: 18 токенов → 1.8 токена/слово
RuBERT: 18 токенов → 1.8 токена/слово
Достоевский и Шолохов остаются важными фигурами русской литературы.
Слов: 8
Наш BPE: 18 токенов → 2.25 токена/слово
RuBERT: 12 токенов → 1.5 токена/слово
Нейросетевой автопереводчик неожиданно переинтерпретировал исходное сообщение.
Слов: 6
Наш BPE: 23 токенов → 3.83 токена/слово
RuBERT: 17 токенов → 2.83 токена/слово


In [11]:
# сравнение по среднему числу токенов на слово
print("Среднее наш BPE:", round(sum(my_ratios) / len(my_ratios), 2))
print("Среднее RuBERT:", round(sum(rubert_ratios) / len(rubert_ratios), 2))

Среднее наш BPE: 2.84
Среднее RuBERT: 1.97


In [ ]:
# проверка UNK - BPE
print("UNK нашего BPE:", tokenizer.token_to_id("[UNK]"))
for text in test_texts:
    tokens = tokenizer.encode(text).tokens
    unk_tokens = [token for token in tokens if token == "[UNK]"]

    print(text)
    print("UNK:", unk_tokens)

# UNK BPE в тестовой выборке не обранужено

UNK нашего BPE: 1
Мы переосмысливаем недопонятые результаты эксперимента.
UNK: []
Сельскохозяйственный и железнодорожный комплексы требуют модернизации.
UNK: []
Я вчера загуглил это, но ответ оказался каким-то кринжовым.
UNK: []
Достоевский и Шолохов остаются важными фигурами русской литературы.
UNK: []
Нейросетевой автопереводчик неожиданно переинтерпретировал исходное сообщение.
UNK: []


In [ ]:
# проверка UNK - RuBERT
print("UNK RuBERT:", industrial_tokenizer.unk_token)

for text in test_texts:
    tokens = industrial_tokenizer.tokenize(text)
    unk_tokens = [token for token in tokens if token == industrial_tokenizer.unk_token]

    print(text)
    print("UNK:", unk_tokens)

# UNK RuBERT в тестовой выборке не обранужено

UNK RuBERT: [UNK]
Мы переосмысливаем недопонятые результаты эксперимента.
UNK: []
Сельскохозяйственный и железнодорожный комплексы требуют модернизации.
UNK: []
Я вчера загуглил это, но ответ оказался каким-то кринжовым.
UNK: []
Достоевский и Шолохов остаются важными фигурами русской литературы.
UNK: []
Нейросетевой автопереводчик неожиданно переинтерпретировал исходное сообщение.
UNK: []


In [25]:
# 5. Отчёт и анализ.

# Рисуем табличку со сравнениями BPE/RuBERT
results = []

for text in test_texts:
    my_tokens = tokenizer.encode(text).tokens
    rubert_tokens = industrial_tokenizer.tokenize(text)

    results.append({
        "Текст": text,
        "Наш BPE": " | ".join(my_tokens),
        "RuBERT": " | ".join(rubert_tokens),
        "Токенов BPE": len(my_tokens),
        "Токенов RuBERT": len(rubert_tokens)
    })

comparison_df = pd.DataFrame(results)

display(
    comparison_df.style
    .set_properties(subset=["Текст"], **{
        "white-space": "normal",
        "width": "300px"
    })
    .set_properties(subset=["Наш BPE", "RuBERT"], **{
        "white-space": "normal",
        "width": "450px"
    })
)

# Примеры самых удачных и не удачных разбиение токенезатором BPE
# - авто | пере | вод | чик - удачно разбиение, видно разделение на логичные составные части слова
# - пере | ос | мысли | ваем - так же удачное разбиение
# - До | сто | ев | ский / Ш | о | ло | хов - не удачное разбиение, просто много мелких составляющих не несущих никакого смысла

# Выводны по морфологии:
# Само обученый BPE из за ограниченной выборки слаб в том, что он мало видел. Популярные присатвки вроде
# пере- отделяются хорошо, а редки части слов распознгаются тяжелее, это видно по сложным словам из выборок 
# вроде "'хозяйстве', 'нный'"  и именам собственым

# Общий вывод заключается в том, что индустриальные решения учаться на большей выборке, по итогу "видят" больше
# и как следствие точнее дробят текст. Само обученный токенезатор "видел" мало, поэтому лучше работает с теми частями слов, 
# что встречаются чаще и вообще не понмиает, то что никогда не "видел"

,Текст,Наш BPE,RuBERT,Токенов BPE,Токенов RuBERT
0,Мы переосмысливаем недопонятые результаты эксперимента.,Мы | пере | ос | мысли | ваем | недо | поня | тые | результа | ты | эк | спе | ри | мента | .,Мы | переос | ##мысли | ##ваем | недо | ##пон | ##ят | ##ые | результаты | эксперимента | .,15,11
1,Сельскохозяйственный и железнодорожный комплексы требуют модернизации.,С | ель | ско | хозяйстве | нный | и | железнодоро | жный | ком | п | лек | сы | требу | ют | мо | дер | ни | за | ции | .,Сель | ##скохозяй | ##ственный | и | железнодорожный | комплексы | требуют | модернизации | .,20,9
2,"Я вчера загуглил это, но ответ оказался каким-то кринжовым.","Я | вчера | загу | гли | л | это | , | но | ответ | оказался | каким | - | то | кри | н | жо | вым | .","Я | вчера | загу | ##гли | ##л | это | , | но | ответ | оказался | каким | - | то | кри | ##н | ##жо | ##вым | .",18,18
3,Достоевский и Шолохов остаются важными фигурами русской литературы.,До | сто | ев | ский | и | Ш | о | ло | хов | остаются | важ | ными | фигу | рами | русской | литерату | ры | .,Досто | ##евский | и | Ш | ##оло | ##хов | остаются | важными | фигурами | русской | литературы | .,18,12
4,Нейросетевой автопереводчик неожиданно переинтерпретировал исходное сообщение.,Н | ей | ро | се | те | вой | авто | пере | вод | чик | неожиданно | пере | ин | тер | пре | ти | ровал | ис | хо | дное | сообще | ние | .,Ней | ##рос | ##ете | ##вой | автоп | ##ере | ##вод | ##чик | неожиданно | переи | ##нтер | ##пр | ##ети | ##ровал | исходное | сообщение | .,23,17
